#What does groupBy define about an aggregation's output grain?

groupBy() defines the output grain by specifying the grouping columns. The aggregation produces one row for each distinct combination of those columns.
Remember: More grouping columns generally mean a finer grain (more detail), while fewer grouping columns mean a coarser grain (less detail).


#How do count and countDistinct differ?

count(): is used to count the non - null values in a column or count("column_name") is used to exclude rows where that specific column is null

countDistinct():is used to count unique , non null values of a specific column or table .


#How does count(column) treat null values?

count(column) exclude the null values and count only non null values of a specific column

#Why should AOV use orders rather than lines as its denominator?

“AOV uses distinct orders as the denominator because it measures average revenue per order. Using order lines would calculate average revenue per line and could misrepresent customer spending.”

(AOV) - Average revenue per order(Average Order Value)

#Which order status contributes to booked revenue?

Booked revenue is the revenue associated with orders that the business considers successfully booked or confirmed, according to its business rules.

“Booked revenue is calculated by summing the order amounts for the order statuses that qualify under the business definition. For example, if the business considers only confirmed orders as booked, I filter for CONFIRMED status and sum the corresponding revenue. I exclude cancelled orders and handle returned orders according to the agreed revenue-recognition rules.”


#What is cancellation rate's denominator?

The cancellation rate denominator is usually the total number of eligible orders placed, including cancelled orders, so that the metric accurately represents the percentage of orders that were cancelled.

Cancellation Rate =(Distinct Cancelled/OrdersTotal Distinct Orders)*100


#What does sum do when all values are null?

sum() ignores null values, but when all values are null, it returns null. We can use coalesce(sum("amount"), lit(0)) to return zero instead.

#Why can revenue share be different from order-count share?

Revenue share tells us what percentage of total revenue comes from a particular category, product, or order status.
Order-count share tells us what percentage of total orders belongs to that category, product, or order status.

Revenue share and order-count share can differ because the average order value varies across categories or order types. Order-count share measures the proportion of orders, whereas revenue share measures the proportion of monetary value.

Revenue Share(%) = (Revenue from a channel /Total Revenue )* 100

#When should an undefined ratio be null rather than zero?

Ratio =Numerator/ Denominator

An undefined ratio should be NULL when the denominator is zero or when the ratio cannot be meaningfully calculated. We should not automatically replace it with zero because zero and undefined represent different meanings.
- Zero (0) means the calculated result is genuinely zero.
- NULL means the result is unknown, unavailable, or undefined.

“An undefined ratio should be represented as NULL rather than zero when the denominator is zero or the calculation cannot be meaningfully performed. Zero represents a valid calculated result, whereas NULL represents an undefined or unavailable result.


#Why aggregate lines to order grain before calculating order averages?

“We aggregate order lines to order grain before calculating order averages because a single order may contain multiple product lines. If we calculate averages directly at line grain, we measure average line value rather than average order value. By first summing the line amounts for each order, we ensure that every order contributes exactly once to the average and that the metric matches the business definition.”

In [1]:
pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import countDistinct

In [3]:
spark=SparkSession.builder.appName("aggregation").getOrCreate()

#Hands-on questions (11-20)


In [4]:
from decimal import Decimal

order_details = [
    (1001, 5001, 1, 201, "Laptop Stand",         2, Decimal("499.99"),  Decimal("10.50")),
    (1002, 5001, 2, 202, "Wireless Keyboard",    1, Decimal("1299.75"), Decimal("50.25")),
    (1003, 5002, 1, 203, None,                    5, Decimal("99.99"),   Decimal("5.00")),
    (1004, 5002, 2, 204, "Wireless Mouse",       3, Decimal("250.50"), Decimal("15.75")),
    (1001, 5001, 1, 201, "Laptop Stand",         3, Decimal("499.99"),  Decimal("10.50")),
    (1005, 5003, 1, 205, "Webcam",               1, Decimal("999.99"), Decimal("0.00")),
    (1006, 5003, 2, 201, "Laptop Stand",         6, Decimal("499.99"), Decimal("25.50")),
    (1001, 5001, 1, 201, "Laptop ",         2, Decimal("499.99"),  Decimal("10.50")),
    (1007, 5004, 1, 206, None,                   2, Decimal("750.25"), Decimal("20.00")),
    (1008, 5005, 1, 207, "Mechanical Keyboard",   None, Decimal("1599.90"),Decimal("100.50")),
    (1009, 5005, 2, 208, "Mouse Pad",             6, Decimal("75.25"),  Decimal("5.25")),
    (1006, 5003, 2, 201, "Laptop Stand",         7, Decimal("499.99"), Decimal("25.50")),
    (1010, 5006, 1, 209, "USB Hub",               2, Decimal("349.99"), None),
    (1011, 5007, 1, 210, "Headphones",            3, Decimal("450.75"), Decimal("30.25")),
    (1012, 5008, 1, 203, "USB Cable",             2, Decimal("99.99"),  Decimal("10.00")),
    (1013, 5009, 1, 211, "4K Monitor",            1, Decimal("2499.99"),Decimal("150.75")),
    (1014, 5010, 1, 212, "Webcam Cover",          5, Decimal("125.50"), Decimal("12.50")),
    (1001, 5001, 1, 201, "Stand",         8, Decimal("499.99"),  Decimal("10.50")),
    (1015, 5010, 2, 201, "Laptop Stand",          2, Decimal("499.99"), None),
    (1006, 5003, 2, 201, "Laptop Stand",         4, Decimal("499.99"), Decimal("25.50")),
    (1006, 5003, 2, 201, "Laptop Stand",         4, Decimal("499.99"), Decimal("25.50")),
    (1004, 5002, 2, 204, "Wireless",       3, Decimal("250.50"), Decimal("15.75")),
    (1004, 5002, 2, 204, " Mouse",       3, Decimal("250.50"), Decimal("15.75")),
    (1004, 5002, 2, 204, "Wireless Mouse",       3, Decimal("250.50"), Decimal("15.75")),
     (1016, 5001, 3, 213, "USB-C Adapter", 2, Decimal("299.99"), Decimal("15.00")),
    (1017, 5001, 4, 214, "Laptop Sleeve", 1, Decimal("599.99"), Decimal("25.00"))
]
schema=StructType([
    StructField("order_item_id",IntegerType(),True),
    StructField("order_id",IntegerType(),True),
    StructField("line_id",IntegerType(),True),
    StructField("product_id",IntegerType(),True),
    StructField("product_name",StringType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DecimalType(18,2),True),
    StructField("discount",DecimalType(18,2),True)
])
df_order=spark.createDataFrame(order_details,schema)
df_order.show()

+-------------+--------+-------+----------+-------------------+--------+----------+--------+
|order_item_id|order_id|line_id|product_id|       product_name|quantity|unit_price|discount|
+-------------+--------+-------+----------+-------------------+--------+----------+--------+
|         1001|    5001|      1|       201|       Laptop Stand|       2|    499.99|   10.50|
|         1002|    5001|      2|       202|  Wireless Keyboard|       1|   1299.75|   50.25|
|         1003|    5002|      1|       203|               NULL|       5|     99.99|    5.00|
|         1004|    5002|      2|       204|     Wireless Mouse|       3|    250.50|   15.75|
|         1001|    5001|      1|       201|       Laptop Stand|       3|    499.99|   10.50|
|         1005|    5003|      1|       205|             Webcam|       1|    999.99|    0.00|
|         1006|    5003|      2|       201|       Laptop Stand|       6|    499.99|   25.50|
|         1001|    5001|      1|       201|            Laptop |       

In [5]:
#Sum clean line_net by order_id.

line_net=df_order.withColumn("line_net",(col("quantity")*col("unit_price") - col("discount")))
# line_net.show()
clean_data=line_net.filter((col("product_name").isNotNull()) & (col("quantity").isNotNull()) & (col("discount").isNotNull()))
clean_data.show()
total= clean_data.groupBy("order_id").agg(sum(col("line_net")).alias("total_line_net"))
total.show()

+-------------+--------+-------+----------+-----------------+--------+----------+--------+--------+
|order_item_id|order_id|line_id|product_id|     product_name|quantity|unit_price|discount|line_net|
+-------------+--------+-------+----------+-----------------+--------+----------+--------+--------+
|         1001|    5001|      1|       201|     Laptop Stand|       2|    499.99|   10.50|  989.48|
|         1002|    5001|      2|       202|Wireless Keyboard|       1|   1299.75|   50.25| 1249.50|
|         1004|    5002|      2|       204|   Wireless Mouse|       3|    250.50|   15.75|  735.75|
|         1001|    5001|      1|       201|     Laptop Stand|       3|    499.99|   10.50| 1489.47|
|         1005|    5003|      1|       205|           Webcam|       1|    999.99|    0.00|  999.99|
|         1006|    5003|      2|       201|     Laptop Stand|       6|    499.99|   25.50| 2974.44|
|         1001|    5001|      1|       201|          Laptop |       2|    499.99|   10.50|  989.48|


In [6]:
orders = [
    (5001, 101, "2026-09-20", "Completed", "UPI", "Visakhapatnam", "Andhra Pradesh", 12500.50, 750.00, 100.00, "Website"),
    (5002, 102, "2026-09-18", "Completed", "Card", "Hyderabad", "Telangana", 8500.00, 500.00, 150.00, "Mobile App"),
    (5003, 103, "2026-09-15", "Pending", "NetBanking", "Bengaluru", "Karnataka", 6200.75, 300.00, 120.00, "Marketplace"),
    (5004, 104, "2026-09-12", "Completed", "Card", "Mumbai", "Maharashtra", 18500.00, 1200.00, 200.00, "Website"),
    (5005, None, "2026-09-10", "Cancelled", "UPI", "Chennai", "Tamil Nadu", 4500.25, 250.00, 100.00, "Physical Store"),
    (5006, 106, "2026-09-08", "Completed", "Cash", "Vijayawada", "Andhra Pradesh", 3200.00, 150.00, 80.00, "Physical Store"),
    (5007, 107, "2026-09-05", "Returned", "Card", "Pune", "Maharashtra", 9800.50, 600.00, 150.00, "Marketplace"),
    (5008, 108, "2026-09-02", "Completed", "NetBanking", "Delhi", "Delhi", 15200.00, 1000.00, 180.00, "Mobile App"),
    (5009, 109, "2026-08-28", "Pending", "UPI", "Kolkata", "West Bengal", 2750.75, 100.00, 70.00, "Website"),
    (5010, 110, "2026-08-25", "Completed", "Card", "Ahmedabad", "Gujarat", 11200.00, 800.00, 150.00, "Marketplace"),
    (5011, 111, "2026-08-20", "Completed", "UPI", "Kochi", "Kerala", 7350.50, 450.00, 120.00, "Website"),
    (5012, 112, "2026-08-15", "Cancelled", "Cash", "Bhubaneswar", "Odisha", 3900.00, 200.00, 90.00, "Physical Store"),
    (5013, None, "2026-08-10", "Completed", "NetBanking", "Coimbatore", "Tamil Nadu", 16400.25, 950.00, 175.00, "Mobile App"),
    (5014, 114, "2026-08-05", "Returned", "Card", "Tirupati", "Andhra Pradesh", 5600.00, 350.00, 100.00, "Marketplace"),
    (5015, 115, "2026-08-01", "Completed", "UPI", "Jaipur", "Rajasthan", 8900.75, 550.00, 130.00, "Website")
]


order_schema=StructType([
    StructField("order_id",IntegerType(),True),
    StructField("customer_id",IntegerType(),True),
    StructField("date",StringType(),True),
    StructField("order_status",StringType(),True),
    StructField("payment_method",StringType(),True),
    StructField("city",StringType(),True),
    StructField("state",StringType(),True),
    StructField("order_amount",DoubleType(),True),
    StructField("shipping_fee",DoubleType(),True),
    StructField("discount",DoubleType(),True),
    StructField("channel",StringType(),True)

])
df_order_details=spark.createDataFrame(orders,order_schema)
df_order_details=df_order_details.withColumn("date",col("date").cast("date"))
df_order_details.show()
df_order_details.printSchema()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|order_id|customer_id|      date|order_status|payment_method|         city|         state|order_amount|shipping_fee|discount|       channel|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|    5001|        101|2026-09-20|   Completed|           UPI|Visakhapatnam|Andhra Pradesh|     12500.5|       750.0|   100.0|       Website|
|    5002|        102|2026-09-18|   Completed|          Card|    Hyderabad|     Telangana|      8500.0|       500.0|   150.0|    Mobile App|
|    5003|        103|2026-09-15|     Pending|    NetBanking|    Bengaluru|     Karnataka|     6200.75|       300.0|   120.0|   Marketplace|
|    5004|        104|2026-09-12|   Completed|          Card|       Mumbai|   Maharashtra|     18500.0|      1200.0|   200.0|       Website|
|    5005|   

In [7]:
#Calculate booked revenue for completed orders only.
df_order_completed = df_order_details.filter(col("order_status")=="Completed")
df_order_completed.show()
calculate_revenue = df_order_completed.withColumn("revenue",(col("order_amount")) - (col("discount")))
calculate_revenue.show()
total_booked_revenue=calculate_revenue.agg(sum("revenue").alias("total_booked_revenue"))
total_booked_revenue.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|order_id|customer_id|      date|order_status|payment_method|         city|         state|order_amount|shipping_fee|discount|       channel|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|    5001|        101|2026-09-20|   Completed|           UPI|Visakhapatnam|Andhra Pradesh|     12500.5|       750.0|   100.0|       Website|
|    5002|        102|2026-09-18|   Completed|          Card|    Hyderabad|     Telangana|      8500.0|       500.0|   150.0|    Mobile App|
|    5004|        104|2026-09-12|   Completed|          Card|       Mumbai|   Maharashtra|     18500.0|      1200.0|   200.0|       Website|
|    5006|        106|2026-09-08|   Completed|          Cash|   Vijayawada|Andhra Pradesh|      3200.0|       150.0|    80.0|Physical Store|
|    5008|   

In [8]:
# Compute store revenue and distinct completed-order counts.
df_order_completed=df_order_details.filter(col("order_status")=="Completed")
df_order_completed.show()
calculate_revenue=df_order_completed.withColumn("revenue",col("order_amount")-col("discount"))
grouping=calculate_revenue.groupBy("city").agg(F.sum("revenue").alias("total_revenue"),F.countDistinct("order_id").alias("unique_count"))
grouping.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|order_id|customer_id|      date|order_status|payment_method|         city|         state|order_amount|shipping_fee|discount|       channel|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|    5001|        101|2026-09-20|   Completed|           UPI|Visakhapatnam|Andhra Pradesh|     12500.5|       750.0|   100.0|       Website|
|    5002|        102|2026-09-18|   Completed|          Card|    Hyderabad|     Telangana|      8500.0|       500.0|   150.0|    Mobile App|
|    5004|        104|2026-09-12|   Completed|          Card|       Mumbai|   Maharashtra|     18500.0|      1200.0|   200.0|       Website|
|    5006|        106|2026-09-08|   Completed|          Cash|   Vijayawada|Andhra Pradesh|      3200.0|       150.0|    80.0|Physical Store|
|    5008|   

In [9]:
#Calculate store AOV from revenue and completed-order count.
df_order_completed=df_order_details.filter(col("order_status")=="Completed")
# df_order_completed.show()
df_revenue=df_order_completed.withColumn("revenue",col("order_amount")-col("discount"))
# df_revenue.show()
df_total_revenue=df_revenue.groupBy("city").agg(sum("revenue").alias("total_revenue"),countDistinct("order_id").alias("unique_count"))
df_total_revenue.show()
df_AOV=df_total_revenue.withColumn("AOV",col("total_revenue")/col("unique_count"))

df_AOV.show()

+-------------+-------------+------------+
|         city|total_revenue|unique_count|
+-------------+-------------+------------+
|        Kochi|       7230.5|           1|
|       Mumbai|      18300.0|           1|
|    Ahmedabad|      11050.0|           1|
|        Delhi|      15020.0|           1|
|   Coimbatore|     16225.25|           1|
|    Hyderabad|       8350.0|           1|
|   Vijayawada|       3120.0|           1|
|       Jaipur|      8770.75|           1|
|Visakhapatnam|      12400.5|           1|
+-------------+-------------+------------+

+-------------+-------------+------------+--------+
|         city|total_revenue|unique_count|     AOV|
+-------------+-------------+------------+--------+
|        Kochi|       7230.5|           1|  7230.5|
|       Mumbai|      18300.0|           1| 18300.0|
|    Ahmedabad|      11050.0|           1| 11050.0|
|        Delhi|      15020.0|           1| 15020.0|
|   Coimbatore|     16225.25|           1|16225.25|
|    Hyderabad|       83

In [10]:
# Calculate cancellation rate at order grain.
df_cancel=df_order_details.filter(col("order_status")=="Cancelled")
df_cancel.show()
count_orders=df_order_details.count()
count_cancel=df_cancel.count()
print(count_orders)
cancel_rate=(count_cancel/count_orders)*100
cancel_rate


+--------+-----------+----------+------------+--------------+-----------+----------+------------+------------+--------+--------------+
|order_id|customer_id|      date|order_status|payment_method|       city|     state|order_amount|shipping_fee|discount|       channel|
+--------+-----------+----------+------------+--------------+-----------+----------+------------+------------+--------+--------------+
|    5005|       NULL|2026-09-10|   Cancelled|           UPI|    Chennai|Tamil Nadu|     4500.25|       250.0|   100.0|Physical Store|
|    5012|        112|2026-08-15|   Cancelled|          Cash|Bhubaneswar|    Odisha|      3900.0|       200.0|    90.0|Physical Store|
+--------+-----------+----------+------------+--------------+-----------+----------+------------+------------+--------+--------------+

15


13.333333333333334

In [11]:
df_order_details.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|order_id|customer_id|      date|order_status|payment_method|         city|         state|order_amount|shipping_fee|discount|       channel|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|    5001|        101|2026-09-20|   Completed|           UPI|Visakhapatnam|Andhra Pradesh|     12500.5|       750.0|   100.0|       Website|
|    5002|        102|2026-09-18|   Completed|          Card|    Hyderabad|     Telangana|      8500.0|       500.0|   150.0|    Mobile App|
|    5003|        103|2026-09-15|     Pending|    NetBanking|    Bengaluru|     Karnataka|     6200.75|       300.0|   120.0|   Marketplace|
|    5004|        104|2026-09-12|   Completed|          Card|       Mumbai|   Maharashtra|     18500.0|      1200.0|   200.0|       Website|
|    5005|   

In [12]:
# Compute channel revenue shares and verify their total.
# Revenue Share(%) = (Revenue from a channel /Total Revenue )* 100
revenue=df_order_details.withColumn("revenue",col("order_amount")-col("discount"))
revenue.show()
total_revenue=revenue.agg(sum("revenue"))
total_revenue.show()
channel_revenue=revenue.groupBy("channel").agg(sum("revenue").alias("channel_revenue"))
channel_revenue.show()
revenue_share=channel_revenue.withColumn("revenue_share",(col("channel_revenue")/total_revenue.first()[0]) * 100)
revenue_share.show()




+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+--------+
|order_id|customer_id|      date|order_status|payment_method|         city|         state|order_amount|shipping_fee|discount|       channel| revenue|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+--------+
|    5001|        101|2026-09-20|   Completed|           UPI|Visakhapatnam|Andhra Pradesh|     12500.5|       750.0|   100.0|       Website| 12400.5|
|    5002|        102|2026-09-18|   Completed|          Card|    Hyderabad|     Telangana|      8500.0|       500.0|   150.0|    Mobile App|  8350.0|
|    5003|        103|2026-09-15|     Pending|    NetBanking|    Bengaluru|     Karnataka|     6200.75|       300.0|   120.0|   Marketplace| 6080.75|
|    5004|        104|2026-09-12|   Completed|          Card|       Mumbai|   Maharashtra|     18500

In [13]:
# Count orders containing more than three accepted lines.
# df_order.show()
business_keys=["order_id","line_id"]
df_deduplicates=df_order.dropDuplicates(business_keys)
df_deduplicates.show()
df_count=df_deduplicates.groupBy("order_id").agg(count("line_id").alias("no_of_records")).filter(col("no_of_records")>3)
df_count.show()

+-------------+--------+-------+----------+-------------------+--------+----------+--------+
|order_item_id|order_id|line_id|product_id|       product_name|quantity|unit_price|discount|
+-------------+--------+-------+----------+-------------------+--------+----------+--------+
|         1001|    5001|      1|       201|       Laptop Stand|       2|    499.99|   10.50|
|         1002|    5001|      2|       202|  Wireless Keyboard|       1|   1299.75|   50.25|
|         1016|    5001|      3|       213|      USB-C Adapter|       2|    299.99|   15.00|
|         1017|    5001|      4|       214|      Laptop Sleeve|       1|    599.99|   25.00|
|         1003|    5002|      1|       203|               NULL|       5|     99.99|    5.00|
|         1004|    5002|      2|       204|     Wireless Mouse|       3|    250.50|   15.75|
|         1005|    5003|      1|       205|             Webcam|       1|    999.99|    0.00|
|         1006|    5003|      2|       201|       Laptop Stand|       

In [14]:
df_order.show()

+-------------+--------+-------+----------+-------------------+--------+----------+--------+
|order_item_id|order_id|line_id|product_id|       product_name|quantity|unit_price|discount|
+-------------+--------+-------+----------+-------------------+--------+----------+--------+
|         1001|    5001|      1|       201|       Laptop Stand|       2|    499.99|   10.50|
|         1002|    5001|      2|       202|  Wireless Keyboard|       1|   1299.75|   50.25|
|         1003|    5002|      1|       203|               NULL|       5|     99.99|    5.00|
|         1004|    5002|      2|       204|     Wireless Mouse|       3|    250.50|   15.75|
|         1001|    5001|      1|       201|       Laptop Stand|       3|    499.99|   10.50|
|         1005|    5003|      1|       205|             Webcam|       1|    999.99|    0.00|
|         1006|    5003|      2|       201|       Laptop Stand|       6|    499.99|   25.50|
|         1001|    5001|      1|       201|            Laptop |       

In [15]:
# Find the highest-revenue product by units and by money.
df_revenue=df_order.withColumn("revenue", (col("quantity")*col("unit_price"))-col("discount"))
# df_revenue.show()
df_grouping=df_revenue.groupBy("product_id","product_name").agg(sum("revenue").alias("revenue_per_product")).orderBy(col("revenue_per_product").desc())
# df_grouping.show()
print("Highest  revenue product:",df_grouping.first())
df_grouping_units=df_revenue.groupBy("product_id","product_name").agg(sum("quantity").alias("total_units")).orderBy(col("total_units").desc())
# df_grouping_units.show()
print("Highest product units sold:",df_grouping_units.first())

Highest  revenue product: Row(product_id=201, product_name='Laptop Stand', revenue_per_product=Decimal('12876.74'))
Highest product units sold: Row(product_id=201, product_name='Laptop Stand', total_units=28)


In [16]:
# Use conditional aggregation for completed and cancelled order counts.
# df_order_details.show()
df_completed=df_order_details.agg(sum(when(col("order_status")=="Completed",1).otherwise(0)).alias("completed_count"),
                                   sum(when(col("order_status")=="Cancelled",1).otherwise(0)).alias("cancelled_count"))
df_completed.show()



+---------------+---------------+
|completed_count|cancelled_count|
+---------------+---------------+
|              9|              2|
+---------------+---------------+



In [17]:
store_revenue = [

    (2001, 101, 301, "Laptop",          "Electronics", 2, Decimal("65000.00"), Decimal("1500.00"), Decimal("52000.00")),
    (2002, 101, 302, "Wireless Mouse",  "Accessories", 5, Decimal("750.50"),   Decimal("50.00"),   Decimal("500.00")),
    (2003, 102, 303, "Office Chair",    "Furniture",   3, Decimal("5500.00"),  Decimal("300.00"),  Decimal("4000.00")),
    (2004, 103, 304, "Desk Lamp",       "Lighting",    4, Decimal("1250.75"), Decimal("100.00"),  Decimal("850.00")),
    (2005, 101, 301, "Laptop",          "Electronics", 1, Decimal("65000.00"), Decimal("2000.00"), Decimal("52000.00")),
    (2006, 104, 305, "Keyboard",        "Accessories", 6, Decimal("1800.00"),  Decimal("150.00"),  Decimal("1200.00")),
    (2007, 102, 306, "Monitor",         "Electronics", 2, Decimal("18500.00"), Decimal("500.00"),  Decimal("14500.00")),
    (2008, 103, 307, None,              "Furniture",   2, Decimal("3200.00"),  Decimal("100.00"),  Decimal("2400.00")),
    (2009, 104, 302, "Wireless Mouse",  "Accessories", 8, Decimal("750.50"),   Decimal("75.00"),   Decimal("500.00")),
    (2010, 105, 308, "Headphones",      "Electronics", 3, Decimal("2500.00"),  Decimal("200.00"),  Decimal("1700.00")),
    (2011, 101, 309, "USB Hub",         "Accessories", 5, Decimal("899.99"),   None,               Decimal("600.00")),
    (2012, 102, 310, "Bookshelf",       "Furniture",   None, Decimal("7500.00"), Decimal("500.00"), Decimal("5500.00")),
    (2013, 103, 304, "Desk Lamp",       "Lighting",    4, Decimal("1250.75"), Decimal("100.00"),  Decimal("850.00")),
    (2014, 104, 311, "Smartphone",      "Electronics", 2, Decimal("42000.00"), Decimal("1000.00"), Decimal("35000.00")),
    (2015, 105, 312, "Printer",         "Electronics", 1, Decimal("12500.00"), Decimal("750.00"), Decimal("9800.00")),
]

schema=StructType([
    StructField("sale_id",IntegerType(),False),
    StructField("store_id",IntegerType(),True),
    StructField("product_id",IntegerType(),True),
    StructField("product_name",StringType(),True),
    StructField("category",StringType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DecimalType(10,2),True),
    StructField("discount",DecimalType(10,2),True),
    StructField("cost_price",DecimalType())
])
df_store_revenue=spark.createDataFrame(store_revenue,schema)


In [18]:
global_bookings = [


    (3001, 6001, "India",        "Bengaluru",  "Hotel",  2, Decimal("450.00"),  Decimal("25.00"),  "Confirmed"),
    (3002, 6002, "USA",          "New York",   "Flight", 1, Decimal("850.50"),  Decimal("50.00"),  "Confirmed"),
    (3003, 6003, "UK",           "London",     "Hotel",  3, Decimal("720.00"),  Decimal("40.00"),  "Completed"),
    (3004, 6004, "UAE",          "Dubai",      "Hotel",  2, Decimal("1200.00"), Decimal("100.00"), "Confirmed"),
    (3005, 6005, "Singapore",    "Singapore",  "Flight", 2, Decimal("560.00"),  Decimal("30.00"),  "Cancelled"),
    (3006, 6001, "India",        "Mumbai",     "Flight", 2, Decimal("320.00"),  Decimal("20.00"),  "Completed"),
    (3007, 6006, "Australia",    "Sydney",     "Hotel",  4, Decimal("980.75"),  Decimal("75.00"),  "Confirmed"),
    (3008, 6007, "Japan",        "Tokyo",      "Hotel",  2, Decimal("650.00"),  Decimal("50.00"),  "Pending"),
    (3009, 6008, "France",       "Paris",      "Flight", 1, Decimal("430.00"),  Decimal("25.00"),  "Completed"),
    (3010, 6009, "Germany",      "Berlin",     "Hotel",  2, Decimal("580.00"),  Decimal("35.00"),  "Confirmed"),
    (3011, 6010, "Canada",       "Toronto",    "Flight", 3, Decimal("1100.00"), None,               "Confirmed"),
    (3012, 6011, "India",        "Delhi",      "Hotel",  None, Decimal("275.00"), Decimal("15.00"), "Pending"),
    (3013, 6012, "Italy",        "Rome",       "Hotel",  2, Decimal("690.00"),  Decimal("45.00"),  "Completed"),
    (3014, 6013, "Spain",        "Barcelona",  "Flight", 2, Decimal("510.00"),  Decimal("30.00"),  "Cancelled"),
    (3015, 6014, "Thailand",     "Bangkok",    "Hotel",  3, Decimal("390.00"),  Decimal("20.00"),  "Confirmed")
]

schema=StructType([
    StructField("booking_id",IntegerType(),False),
    StructField("customer_id",IntegerType(),True),
    StructField("country",StringType(),True),
    StructField("city",StringType(),True),
    StructField("booking_type",StringType(),True),
    StructField("guests",IntegerType(),True),
    StructField("booking_amount",DecimalType(10,2),True),
    StructField("discount",DecimalType(10,2)),
    StructField("booking_status",StringType(),True)
])
df_globa_booking=spark.createDataFrame(global_bookings,schema)

In [19]:
# Reconcile summed store revenue with global booked revenue.
df_store_net_revenue = df_store_revenue.withColumn("net_revenue",(col("quantity")*col("unit_price")) - col("discount"))
df_store_net_revenue.show()
df_total_revenue=df_store_net_revenue.agg(sum(col("net_revenue")).alias("store_total"))
df_total_revenue.show()
# df_distinct=df_globa_booking.distinct()
# df_distinct.show()
df_global_revenue=df_globa_booking.filter(col("booking_status")=="Completed")
# df_global_revenue.show()
df_global_net_revenue=df_global_revenue.withColumn("net_revenue",col("booking_amount")-col("discount"))
df_global_net_revenue.show()
df_total_global_revenue=df_global_net_revenue.agg(sum(col("net_revenue")).alias("total_global_revenue"))
df_total_global_revenue.show()

df_reconcilation=df_total_revenue.join(df_total_global_revenue,how="cross")
df_reconcilation=df_reconcilation.withColumn("revenue_difference",col("store_total")-col("total_global_revenue"))
df_reconcilation.show()

+-------+--------+----------+--------------+-----------+--------+----------+--------+----------+-----------+
|sale_id|store_id|product_id|  product_name|   category|quantity|unit_price|discount|cost_price|net_revenue|
+-------+--------+----------+--------------+-----------+--------+----------+--------+----------+-----------+
|   2001|     101|       301|        Laptop|Electronics|       2|  65000.00| 1500.00|     52000|  128500.00|
|   2002|     101|       302|Wireless Mouse|Accessories|       5|    750.50|   50.00|       500|    3702.50|
|   2003|     102|       303|  Office Chair|  Furniture|       3|   5500.00|  300.00|      4000|   16200.00|
|   2004|     103|       304|     Desk Lamp|   Lighting|       4|   1250.75|  100.00|       850|    4903.00|
|   2005|     101|       301|        Laptop|Electronics|       1|  65000.00| 2000.00|     52000|   63000.00|
|   2006|     104|       305|      Keyboard|Accessories|       6|   1800.00|  150.00|      1200|   10650.00|
|   2007|     102| 

#Advanced challenges (21-30)


In [20]:
store_order = [
    (5001, 101, "2026-09-20", "Completed", "UPI", "Visakhapatnam", "Andhra Pradesh", 12500.50, 750.00, 100.00, "Website", 201),
    (5002, 102, "2026-09-18", "Completed", "Card", "Hyderabad", "Telangana", 8500.00, 500.00, 150.00, "Mobile App", 202),
    (5003, 103, "2026-09-15", "Pending", "NetBanking", "Bengaluru", "Karnataka", 6200.75, 300.00, 120.00, "Marketplace", 203),
    (5004, 104, "2026-09-12", "Completed", "Card", "Mumbai", "Maharashtra", 18500.00, 1200.00, 200.00, "Website", 204),
    (5005, 105, "2026-09-10", "Cancelled", "UPI", "Chennai", "Tamil Nadu", 4500.25, 250.00, 100.00, "Physical Store", 205),
    (5006, 106, "2026-09-08", "Completed", "Cash", "Vijayawada", "Andhra Pradesh", 3200.00, 150.00, 80.00, "Physical Store", 206),
    (5007, 107, "2026-09-05", "Returned", "Card", "Pune", "Maharashtra", 9800.50, 600.00, 150.00, "Marketplace", 201),
    (5008, 108, "2026-09-02", "Completed", "NetBanking", "Delhi", "Delhi", 15200.00, 1000.00, 180.00, "Mobile App", 208),
    (5009, 109, "2026-08-28", "Pending", "UPI", "Kolkata", "West Bengal", 2750.75, 100.00, 70.00, "Website", 202),
    (5010, 110, "2026-08-25", "Completed", "Card", "Ahmedabad", "Gujarat", 11200.00, 800.00, 150.00, "Marketplace", 210),
    (5011, 111, "2026-08-20", "Completed", "UPI", "Kochi", "Kerala", 7350.50, 450.00, 120.00, "Website", 211),
    (5012, 112, "2026-08-15", "Cancelled", "Cash", "Bhubaneswar", "Odisha", 3900.00, 200.00, 90.00, "Physical Store", 212),
    (5013, 113, "2026-08-10", "Completed", "NetBanking", "Coimbatore", "Tamil Nadu", 16400.25, 950.00, 175.00, "Mobile App", 213),
    (5014, 114, "2026-08-05", "Returned", "Card", "Tirupati", "Andhra Pradesh", 5600.00, 350.00, 100.00, "Marketplace", 214),
    (5015, 115, "2026-08-01", "Completed", "UPI", "Jaipur", "Rajasthan", 8900.75, 550.00, 130.00, "Website", 215)
]
columns = [
    "order_id", "customer_id", "order_date", "booking_status",
    "payment_method", "city", "state", "booking_amount",
    "discount", "shipping_fee", "sales_channel", "store_id"
]

df_store=spark.createDataFrame(store_order,columns)



In [21]:
df_store.show()

+--------+-----------+----------+--------------+--------------+-------------+--------------+--------------+--------+------------+--------------+--------+
|order_id|customer_id|order_date|booking_status|payment_method|         city|         state|booking_amount|discount|shipping_fee| sales_channel|store_id|
+--------+-----------+----------+--------------+--------------+-------------+--------------+--------------+--------+------------+--------------+--------+
|    5001|        101|2026-09-20|     Completed|           UPI|Visakhapatnam|Andhra Pradesh|       12500.5|   750.0|       100.0|       Website|     201|
|    5002|        102|2026-09-18|     Completed|          Card|    Hyderabad|     Telangana|        8500.0|   500.0|       150.0|    Mobile App|     202|
|    5003|        103|2026-09-15|       Pending|    NetBanking|    Bengaluru|     Karnataka|       6200.75|   300.0|       120.0|   Marketplace|     203|
|    5004|        104|2026-09-12|     Completed|          Card|       Mumbai

In [22]:
# Create a store with no completed orders and define its output metrics.
df_count=df_store.groupBy("store_id").agg(sum(when(col("booking_status")=="Completed",1).otherwise(0)).alias("order_count"))
# df_count.show()
df_filter=df_count.filter(col("order_count")==0)
df_filter.show()
df_output=df_store.groupBy("store_id").agg(sum(when(col("booking_status")=="Completed",(col("booking_amount")-col("discount"))).otherwise(0)).alias("complete_revenue"))
df_output.show()

df_final=df_filter.join(df_output,on="store_id",how="left")
df_final = df_final.withColumn( "completed_orders", col("order_count"))\
                   .withColumn("completed_revenue", col("complete_revenue"))\
                   .withColumn("store_status", when(col("order_count") == 0, "No Completed Orders")
                    .otherwise("Completed Orders Available"))
df_final.select("store_id","completed_orders","completed_revenue","store_status").show(truncate=False)

+--------+-----------+
|store_id|order_count|
+--------+-----------+
|     203|          0|
|     205|          0|
|     212|          0|
|     214|          0|
+--------+-----------+

+--------+----------------+
|store_id|complete_revenue|
+--------+----------------+
|     202|          8000.0|
|     201|         11750.5|
|     203|             0.0|
|     205|             0.0|
|     206|          3050.0|
|     204|         17300.0|
|     212|             0.0|
|     215|         8350.75|
|     208|         14200.0|
|     214|             0.0|
|     210|         10400.0|
|     213|        15450.25|
|     211|          6900.5|
+--------+----------------+

+--------+----------------+-----------------+-------------------+
|store_id|completed_orders|completed_revenue|store_status       |
+--------+----------------+-----------------+-------------------+
|203     |0               |0.0              |No Completed Orders|
|205     |0               |0.0              |No Completed Orders|
|212    

#Explain why averaging store AOV values does not produce global AOV.

“Taking a simple average of store AOVs does not generally produce the global AOV because stores have different numbers of orders. Global AOV must be calculated as total revenue divided by total orders across all stores. This correctly weights each store according to its order volume.”



In [23]:
df_store.show()

+--------+-----------+----------+--------------+--------------+-------------+--------------+--------------+--------+------------+--------------+--------+
|order_id|customer_id|order_date|booking_status|payment_method|         city|         state|booking_amount|discount|shipping_fee| sales_channel|store_id|
+--------+-----------+----------+--------------+--------------+-------------+--------------+--------------+--------+------------+--------------+--------+
|    5001|        101|2026-09-20|     Completed|           UPI|Visakhapatnam|Andhra Pradesh|       12500.5|   750.0|       100.0|       Website|     201|
|    5002|        102|2026-09-18|     Completed|          Card|    Hyderabad|     Telangana|        8500.0|   500.0|       150.0|    Mobile App|     202|
|    5003|        103|2026-09-15|       Pending|    NetBanking|    Bengaluru|     Karnataka|       6200.75|   300.0|       120.0|   Marketplace|     203|
|    5004|        104|2026-09-12|     Completed|          Card|       Mumbai

In [24]:
# Reconstruct global AOV using correctly weighted store metrics.
# complete=df_store.filter(col("booking_status")=="Completed")
# complete.show()
revenue=df_store.groupBy("store_id").agg(sum(when(col("booking_status")=="Completed",col("booking_amount")-col("discount")).otherwise(0)).alias("completed_revenue"))
# revenue.show()

net_revenue=df_store.groupBy("store_id").agg(sum(when(col("booking_status")=="Completed",1).otherwise(0)).alias("completed_orders"))
# net_revenue.show()
df_store_metric= revenue.join(net_revenue,on="store_id",how="inner")
# df_store_metric.show()
store_aov = df_store_metric.withColumn("store_aov",(col("completed_revenue")/when(col("completed_orders")>0,col("completed_orders")).otherwise(None)))
store_aov.show()
global_aov = store_aov.agg(round(sum("completed_revenue")/sum("completed_orders"),2).alias("global_aov"))
global_aov.show()




+--------+-----------------+----------------+---------+
|store_id|completed_revenue|completed_orders|store_aov|
+--------+-----------------+----------------+---------+
|     202|           8000.0|               1|   8000.0|
|     201|          11750.5|               1|  11750.5|
|     203|              0.0|               0|     NULL|
|     205|              0.0|               0|     NULL|
|     206|           3050.0|               1|   3050.0|
|     204|          17300.0|               1|  17300.0|
|     212|              0.0|               0|     NULL|
|     215|          8350.75|               1|  8350.75|
|     208|          14200.0|               1|  14200.0|
|     214|              0.0|               0|     NULL|
|     210|          10400.0|               1|  10400.0|
|     213|         15450.25|               1| 15450.25|
|     211|           6900.5|               1|   6900.5|
+--------+-----------------+----------------+---------+

+----------+
|global_aov|
+----------+
|  10600

In [25]:
df_order_details.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|order_id|customer_id|      date|order_status|payment_method|         city|         state|order_amount|shipping_fee|discount|       channel|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|    5001|        101|2026-09-20|   Completed|           UPI|Visakhapatnam|Andhra Pradesh|     12500.5|       750.0|   100.0|       Website|
|    5002|        102|2026-09-18|   Completed|          Card|    Hyderabad|     Telangana|      8500.0|       500.0|   150.0|    Mobile App|
|    5003|        103|2026-09-15|     Pending|    NetBanking|    Bengaluru|     Karnataka|     6200.75|       300.0|   120.0|   Marketplace|
|    5004|        104|2026-09-12|   Completed|          Card|       Mumbai|   Maharashtra|     18500.0|      1200.0|   200.0|       Website|
|    5005|   

In [26]:
df_order.show()

+-------------+--------+-------+----------+-------------------+--------+----------+--------+
|order_item_id|order_id|line_id|product_id|       product_name|quantity|unit_price|discount|
+-------------+--------+-------+----------+-------------------+--------+----------+--------+
|         1001|    5001|      1|       201|       Laptop Stand|       2|    499.99|   10.50|
|         1002|    5001|      2|       202|  Wireless Keyboard|       1|   1299.75|   50.25|
|         1003|    5002|      1|       203|               NULL|       5|     99.99|    5.00|
|         1004|    5002|      2|       204|     Wireless Mouse|       3|    250.50|   15.75|
|         1001|    5001|      1|       201|       Laptop Stand|       3|    499.99|   10.50|
|         1005|    5003|      1|       205|             Webcam|       1|    999.99|    0.00|
|         1006|    5003|      2|       201|       Laptop Stand|       6|    499.99|   25.50|
|         1001|    5001|      1|       201|            Laptop |       

In [27]:
# Compare count(*) and count(customer_id) on a null-containing fixture.
count_all=df_order_details.agg(count("*").alias("count"))
count_all.show()
count_customer=df_order_details.agg(count("customer_id").alias("customer_count"))
count_customer.show()

+-----+
|count|
+-----+
|   15|
+-----+

+--------------+
|customer_count|
+--------------+
|            13|
+--------------+



In [28]:
# Show how joining order totals back to lines can duplicate order-level revenue.
lines_net=df_order.withColumn("line_net",(col("quantity")*col("unit_price"))-col("discount"))
# lines_net.show(lines_net.count(),truncate=False)
order_revenue=lines_net.groupBy("order_id").agg(sum("line_net").alias("order_revenue"))
df_order_sum=df_order.agg(sum((col("quantity")*col("unit_price"))-col("discount")).alias("line_revenue"))
df_order_sum.show()
df_joins=df_order.join(order_revenue,on="order_id",how="left")
df_joins.show()
df_revenue=df_joins.agg(sum("order_revenue").alias("total_revenue"))
df_revenue.show()

+------------+
|line_revenue|
+------------+
|    31106.02|
+------------+

+--------+-------------+-------+----------+-------------------+--------+----------+--------+-------------+
|order_id|order_item_id|line_id|product_id|       product_name|quantity|unit_price|discount|order_revenue|
+--------+-------------+-------+----------+-------------------+--------+----------+--------+-------------+
|    5006|         1010|      1|       209|            USB Hub|       2|    349.99|    NULL|         NULL|
|    5003|         1005|      1|       205|             Webcam|       1|    999.99|    0.00|     11397.78|
|    5003|         1006|      2|       201|       Laptop Stand|       6|    499.99|   25.50|     11397.78|
|    5003|         1006|      2|       201|       Laptop Stand|       7|    499.99|   25.50|     11397.78|
|    5001|         1001|      1|       201|       Laptop Stand|       2|    499.99|   10.50|      9867.32|
|    5001|         1002|      2|       202|  Wireless Keyboard|     

#Define how an order with all lines rejected affects AOV and quality metrics.

“An order with all lines rejected contributes no accepted revenue and should generally be excluded from AOV calculations when AOV is based on eligible orders. However, it must still be included in data-quality monitoring. I would track fully rejected orders, line rejection rates, and order rejection rates separately. The AOV denominator should follow clearly defined business rules to avoid misleading revenue metrics.”


In [29]:
# Compare approximate distinct counting with exact counts for financial reporting.
df_order_details.show()
distinct_count = df_order_details.agg(countDistinct("customer_id").alias("column_count"))
distinct_count.show()
approx_count=df_order_details.agg(approx_count_distinct("customer_id").alias("approx_count"))
approx_count.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|order_id|customer_id|      date|order_status|payment_method|         city|         state|order_amount|shipping_fee|discount|       channel|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|    5001|        101|2026-09-20|   Completed|           UPI|Visakhapatnam|Andhra Pradesh|     12500.5|       750.0|   100.0|       Website|
|    5002|        102|2026-09-18|   Completed|          Card|    Hyderabad|     Telangana|      8500.0|       500.0|   150.0|    Mobile App|
|    5003|        103|2026-09-15|     Pending|    NetBanking|    Bengaluru|     Karnataka|     6200.75|       300.0|   120.0|   Marketplace|
|    5004|        104|2026-09-12|   Completed|          Card|       Mumbai|   Maharashtra|     18500.0|      1200.0|   200.0|       Website|
|    5005|   

In [30]:
recorded_data = [
    (5001, Decimal("9867.32")),
    (5002, Decimal("3400.00")),
    (5003, Decimal("11397.78")),
    (5004, Decimal("1480.50")),
    (5005, Decimal("446.25")),
    (5007, Decimal("1300.00")),
    (5008, Decimal("189.98")),
    (5009, Decimal("2349.24")),
    (5010, Decimal("615.00"))
]
schema=StructType([
    StructField("order_id",IntegerType(),True),
    StructField("recorded_data",DecimalType(18,2),True)
])
df_recorded_revenue=spark.createDataFrame(recorded_data,schema)
df_recorded_revenue.show()

+--------+-------------+
|order_id|recorded_data|
+--------+-------------+
|    5001|      9867.32|
|    5002|      3400.00|
|    5003|     11397.78|
|    5004|      1480.50|
|    5005|       446.25|
|    5007|      1300.00|
|    5008|       189.98|
|    5009|      2349.24|
|    5010|       615.00|
+--------+-------------+



In [31]:
# Design a revenue reconciliation with decimal precision rather than floating tolerance.
filter_valid=df_order.filter((col("quantity").isNotNull()) & col("discount").isNotNull())
# filter_valid.show()
revenue = filter_valid.withColumn("revenue",(col("quantity")* col("unit_price"))-col("discount"))
# revenue.show()
order_revenue=revenue.groupBy("order_id").agg(sum("revenue").alias("order_revenue"))
# order_revenue.show()
# revenue.printSchema()
df_join=order_revenue.join(df_recorded_revenue,on="order_id",how="inner")
df_join.show()
df_reconcilation=df_join.withColumn("difference",col("order_revenue")-col("recorded_data"))\
                        .withColumn("status",when(col("difference")==lit(Decimal("0.00")),"Reconciled").otherwise("Mismatch"))

df_reconcilation.show()


+--------+-------------+-------------+
|order_id|order_revenue|recorded_data|
+--------+-------------+-------------+
|    5003|     11397.78|     11397.78|
|    5001|      9867.32|      9867.32|
|    5004|      1480.50|      1480.50|
|    5002|      3437.95|      3400.00|
|    5010|       615.00|       615.00|
|    5008|       189.98|       189.98|
|    5007|      1322.00|      1300.00|
|    5005|       446.25|       446.25|
|    5009|      2349.24|      2349.24|
+--------+-------------+-------------+

+--------+-------------+-------------+----------+----------+
|order_id|order_revenue|recorded_data|difference|    status|
+--------+-------------+-------------+----------+----------+
|    5003|     11397.78|     11397.78|      0.00|Reconciled|
|    5001|      9867.32|      9867.32|      0.00|Reconciled|
|    5004|      1480.50|      1480.50|      0.00|Reconciled|
|    5002|      3437.95|      3400.00|     37.95|  Mismatch|
|    5010|       615.00|       615.00|      0.00|Reconciled|
|   

In [32]:
df_order_details.show()

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|order_id|customer_id|      date|order_status|payment_method|         city|         state|order_amount|shipping_fee|discount|       channel|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+
|    5001|        101|2026-09-20|   Completed|           UPI|Visakhapatnam|Andhra Pradesh|     12500.5|       750.0|   100.0|       Website|
|    5002|        102|2026-09-18|   Completed|          Card|    Hyderabad|     Telangana|      8500.0|       500.0|   150.0|    Mobile App|
|    5003|        103|2026-09-15|     Pending|    NetBanking|    Bengaluru|     Karnataka|     6200.75|       300.0|   120.0|   Marketplace|
|    5004|        104|2026-09-12|   Completed|          Card|       Mumbai|   Maharashtra|     18500.0|      1200.0|   200.0|       Website|
|    5005|   

In [33]:
# Inspect the shuffle introduced by grouping and explain its key distribution.
df_revenue=df_order_details.withColumn("revenue",col("order_amount")-col("discount"))
# df_revenue.show()

df_order_details.withColumn("partition_id",col("city")).show()
df_grouped=df_revenue.groupBy(col("city")).agg(count(col("order_id")).alias("total_orders"),sum(col("revenue")).alias("total_revenue"))
df_grouped.show()
df_grouped.explain("formatted")

+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+-------------+
|order_id|customer_id|      date|order_status|payment_method|         city|         state|order_amount|shipping_fee|discount|       channel| partition_id|
+--------+-----------+----------+------------+--------------+-------------+--------------+------------+------------+--------+--------------+-------------+
|    5001|        101|2026-09-20|   Completed|           UPI|Visakhapatnam|Andhra Pradesh|     12500.5|       750.0|   100.0|       Website|Visakhapatnam|
|    5002|        102|2026-09-18|   Completed|          Card|    Hyderabad|     Telangana|      8500.0|       500.0|   150.0|    Mobile App|    Hyderabad|
|    5003|        103|2026-09-15|     Pending|    NetBanking|    Bengaluru|     Karnataka|     6200.75|       300.0|   120.0|   Marketplace|    Bengaluru|
|    5004|        104|2026-09-12|   Completed|          Card|       Mu

Check fixture metric
<!-- from decimal import Decimal

# 1. Expected results calculated independently
expected_revenue = {
    1: Decimal("190.00"),
    2: Decimal("145.00"),
    3: Decimal("19.99")
}

# 2. Check exactly three valid rows remain
actual_count = df_revenue.count()

assert actual_count == 3, (
    f"Expected 3 valid rows, but found {actual_count}"
)

# 3. Check the revenue data type
revenue_type = df_revenue.schema["revenue"].dataType

assert revenue_type.simpleString() == "decimal(10,2)", (
    f"Expected decimal(10,2), but found {revenue_type}"
)

# 4. Collect actual results using order_item_id
actual_rows = {
    row["order_item_id"]: row["revenue"]
    for row in df_revenue.select(
        "order_item_id", "revenue"
    ).collect()
}

# 5. Compare actual results with expected results
assert actual_rows == expected_revenue, (
    f"Revenue mismatch!\n"
    f"Expected: {expected_revenue}\n"
    f"Actual: {actual_rows}"
)

print("All independent fixture assertions passed!") -->

In [70]:
# Write a metric-definition document and independent fixture checks.
# 1. Metric definition
metric_definition = {
    "metric_name": "net_revenue",
    "description": "Quantity multiplied by unit price minus discount",
    "formula": "quantity * unit_price - discount",
    "grain": "one order line",
    "null_policy": "exclude rows with null quantity, unit_price, or discount",
    "data_type": "Decimal(10,2)"
}
print("The Metric Definition:",metric_definition)
df_cast=df_order.withColumn("quantity",col("quantity").cast("Integer"))\
                     .withColumn("unit_price",col("unit_price").cast("Decimal(10,2)"))\
                     .withColumn("discount",col("discount").cast("Decimal(10,2)"))
df_remove_null=df_cast.filter((col("quantity").isNotNull()) & (col("discount").isNotNull()) & (col("unit_price").isNotNull()))
# df_remove_null.show()
df_revenue=df_remove_null.withColumn("revenue",col("quantity")*col("unit_price")-col("discount"))
df_revenue.show()
df_revenue=df_revenue.withColumn("revenue",col("revenue").cast("Decimal(10,2)"))
df_revenue.printSchema()

#independent fixture checks.
data = [
    (1, 5001, 1, 2, Decimal("100.00"), Decimal("10.00")),
    (2, 5001, 2, 3, Decimal("50.00"), Decimal("5.00")),
    (3, 5002, 1, 1, Decimal("19.99"), Decimal("0.00")),
    (4, 5002, 2, None, Decimal("100.00"), Decimal("10.00")),
    (5, 5003, 1, 2, None, Decimal("10.00")),
    (6, 5003, 2, 2, Decimal("100.00"), None)
]

columns = [
    "order_item_id", "order_id", "line_id",
    "quantity", "unit_price", "discount"
]
df_data=spark.createDataFrame(data,columns)
df_cast=df_data.withColumn("quantity",col("quantity").cast("Integer"))\
                     .withColumn("unit_price",col("unit_price").cast("Decimal(10,2)"))\
                     .withColumn("discount",col("discount").cast("Decimal(10,2)"))
df_null=df_cast.filter(col("quantity").isNotNull() & col("unit_price").isNotNull() & col("discount").isNotNull())
# df_null.show()
df_revenue=df_null.withColumn("revenue",col("quantity")*col("unit_price")-col("discount"))
df_revenue.show()
df_revenue=df_revenue.withColumn("revenue",col("revenue").cast("Decimal(10,2)"))
df_revenue.printSchema()
## 2. Check exactly three valid rows remain
actual_count=df_revenue.count()
assert actual_count == 3,( f"Excepted 3 valid rows but found {actual_count}")
# 3. Check the revenue data type
revenue_type = df_revenue.schema["revenue"].dataType
assert revenue_type.simpleString() == "decimal(10,2)", (f"Expected decimal(10,2), but found {revenue_type}")
# 5. Compare actual results with expected results
print("All independent fixture assertions passed!")




The Metric Definition: {'metric_name': 'net_revenue', 'description': 'Quantity multiplied by unit price minus discount', 'formula': 'quantity * unit_price - discount', 'grain': 'one order line', 'null_policy': 'exclude rows with null quantity, unit_price, or discount', 'data_type': 'Decimal(10,2)'}
+-------------+--------+-------+----------+-----------------+--------+----------+--------+-------+
|order_item_id|order_id|line_id|product_id|     product_name|quantity|unit_price|discount|revenue|
+-------------+--------+-------+----------+-----------------+--------+----------+--------+-------+
|         1001|    5001|      1|       201|     Laptop Stand|       2|    499.99|   10.50| 989.48|
|         1002|    5001|      2|       202|Wireless Keyboard|       1|   1299.75|   50.25|1249.50|
|         1003|    5002|      1|       203|             NULL|       5|     99.99|    5.00| 494.95|
|         1004|    5002|      2|       204|   Wireless Mouse|       3|    250.50|   15.75| 735.75|
|      